# Chapter 6 — From Traces to Features

**Book:** *Brain–Computer Interfaces and Cognitive Computing: An Introduction*  
**Prerequisites:** Chapters 0–5.  
**Author:** Er. Rishabh Aryan

Stage S3 and Stage S4 decide what the decoder is allowed to see. A later network cannot recover a band you have filtered out, a trial you have contaminated with the test set, or a neural source you have projected away. This chapter treats preprocessing as design, not as ritual.

Laboratories use MNE-Python, EEGLAB, or vendor stacks. The ideas below are independent of brand. The cells use only NumPy and Matplotlib so the course still runs on a bare interpreter.

**Learning outcomes**

1. Choose a reference, a sampling-aware filter, and an epoching rule and state what each destroys.
2. Name the principal EEG artefact families and an audit that distinguishes them from the claimed neural feature.
3. Apply PCA and a two-class spatial filter in the geometry of Chapter 0.
4. Build time-domain and band-power features matched to the paradigms of Chapter 5.
5. Refuse a feature pipeline that leaks future samples or overlapping windows into training.


---

## 6.1 The pipeline as a sequence of irreversible maps

Write the recorded matrix as $X \in \mathbb{R}^{C \times T}$ (channels $\times$ time), after analogue conversion (Chapter 3). Every later object is a function of $X$:

$$
Z = \Phi_{\text{feat}}\!\left(\Phi_{\text{clean}}(X)\right).
$$

$\Phi_{\text{clean}}$ is referencing, filtering, artefact handling, epoching. $\Phi_{\text{feat}}$ is projection, power, templates, learned embeddings. Both are part of $\theta$ in the broad sense: if you tune them on the test session, you have trained on the test session.

> **Definition 6.1 (Feature).**  
> A computed variable intended to carry the generating mechanism of the chosen paradigm and to be stable enough for Stage S5.

> **Definition 6.2 (Destroyed information).**  
> Any component of $X$ that $\Phi_{\text{clean}}$ or $\Phi_{\text{feat}}$ has made unrecoverable. Destroyed information is not a moral failure. Unnamed destroyed information is.


---

## 6.2 Referencing and montage

EEG voltages are differences. An unstated reference is an unstated experiment.

**Common reference.** All channels versus one site (often mastoid or Cz). Simple. That site’s artefact becomes everyone’s artefact.

**Average reference.** Subtract the mean across a large set of channels. Reasonable when coverage is dense and roughly uniform. Misleading when you have eight forehead electrodes.

**Surface Laplacian / local reference.** Emphasises local radial sources; attenuates broad fields. Can help motor imagery; can hurt a broad P300.

**Bipolar.** Adjacent differences. Robust to common noise; loses absolute topography.

Implanted grids have their own conventions (for example a quiet intracranial reference). The principle is identical: state the montage, and do not change it between training and deployment without recounting $\theta$.

A bad reference can manufacture a “cognitive” signal that is an eye blink at one site copied into every feature.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(6)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

fs, T = 256, 256
t = np.arange(T) / fs
# three channels: neural-like 10 Hz on ch0, blink on ch2, shared line noise
line = 0.4 * np.sin(2 * np.pi * 50 * t)
blink = 3.0 * np.exp(-0.5 * ((t - 0.45) / 0.03) ** 2)
neural = 0.8 * np.sin(2 * np.pi * 10 * t)
X_raw = np.vstack([
    neural + 0.2 * line + 0.1 * np.random.randn(T),
    0.2 * neural + 0.2 * line + 0.1 * np.random.randn(T),
    0.1 * neural + blink + 0.2 * line + 0.1 * np.random.randn(T),
])

X_avg = X_raw - X_raw.mean(axis=0, keepdims=True)
X_car = X_raw - X_raw[2:3, :]  # unfortunate reference on the blinking channel

fig, axes = plt.subplots(3, 1, figsize=(7.6, 5.2), sharex=True)
axes[0].plot(t, X_raw.T)
axes[0].set_title("Raw (arbitrary reference)")
axes[1].plot(t, X_avg.T)
axes[1].set_title("Average reference")
axes[2].plot(t, X_car.T)
axes[2].set_title("Referenced to the blinking channel: blink becomes everyone")
axes[2].set_xlabel("time (s)")
for ax in axes:
    ax.set_ylabel("a.u.")
plt.tight_layout()
plt.show()


---

## 6.3 Sampling, filtering, and group delay

You may not claim a feature above the Nyquist frequency $f_s/2$. Anti-alias filtering belongs in the analogue front-end (Chapter 3). Digital filters after that are design choices.

**High-pass.** Removes slow drift and sweat. A cut-off of 0.1–1 Hz is common. Too high, and a P300 shape is tilted or removed. Too low, and the baseline wanders.

**Low-pass / band-pass.** Motor imagery lives in mu/beta. SSVEP lives in narrow bands around flicker and harmonics. Speech features on cortex may use broader bands. A 40 Hz low-pass that seemed “clean” will destroy the harmonic an SSVEP decoder needed.

**Notch.** 50 or 60 Hz. Use sparingly; a wide notch is a hole in the spectrum you then forget.

**Causal versus non-causal.** Offline zero-phase `filtfilt` looks pretty and is **illegal in a live loop**. It uses the future. For an interface, apply a causal filter and *measure* group delay; put that number into the latency budget of Section 4.4.

> **Definition 6.3 (Causal preprocessing).**  
> A transformation of $x_t$ that is a function only of $x_s$ for $s \le t$, plus parameters frozen before the test block.

Windowed FFT features are causal only if the timestamp of the feature is the *end* of the window (Chapter 5 cell). Labelling a window by its centre while deploying a live system is a silent lie.


In [ ]:
def moving_average_causal(x, k=25):
    '''Crude causal low-pass: group delay roughly (k-1)/2 samples.'''
    k = int(k)
    h = np.ones(k) / k
    y = np.convolve(x, h, mode="full")[: x.size]
    return y


x = neural + 0.3 * np.random.randn(T)
y = moving_average_causal(x, 25)
delay_s = 24 / 2 / fs
print(f"moving-average length 25 samples; approx group delay {delay_s*1000:.1f} ms")

fig, ax = plt.subplots(figsize=(7.6, 2.6))
ax.plot(t, x, color="gray", alpha=0.6, label="raw")
ax.plot(t, y, color="navy", label="causal moving average")
ax.set_xlabel("time (s)")
ax.set_title("Causal smoothing delays the trace; pretty zero-phase would cheat")
ax.legend()
plt.tight_layout()
plt.show()


---

## 6.4 Artefacts: treat them as competing decoders

If a non-neural variable predicts $\hat{y}$ better than your feature, the loop may still work and the scientific sentence is false.

| Family | Look | Why it fools BCI |
|---|---|---|
| Ocular | Slow, frontal, blink spikes | Looks like a state change or an ERP |
| Muscular | Broadband, jaw/neck/forehead | Lives inside mu/beta and gamma |
| Line noise | 50/60 Hz and harmonics | Stable “feature” of a room |
| Motion / electrode pop | Steps, saturation | Class-locked if the user moves on cue |
| Cardiac | Regular ~1 Hz | Rarely the class, sometimes the reference |
| Sweat / drift | Very slow | Breaks baselines mid-block |

**Audit, not folklore.**

1. Record EOG and EMG when you can.  
2. Train a *nuisance decoder* on those channels to predict the same labels.  
3. If the nuisance decoder rivals the EEG decoder, stop claiming a neural class.  
4. Inspect trials time-locked to cues for stereotyped motion.  
5. Never delete artefacts only from the training set; the live loop will meet them.

Independent component analysis can isolate ocular components when coverage is dense. It is not a licence to strip until the class looks perfect. Aggressive cleaning that uses class labels is another leak.


---

## 6.5 Epoching, windows, and leakage

**ERP families** cut epochs locked to stimuli: $[-0.2, 0.8]$ s around each flash is a typical P300 window. Baseline subtraction uses a pre-stimulus interval. If the pre-stimulus interval contains the previous flash’s late wave, you have mixed trials.

**Rhythmic families** use sliding windows. Overlap is allowed in *deployment*. Overlap is forbidden as a way to pretend you have more independent training trials than you do.

**The rule from Chapter 0, now operational.**

- Split by *block* or *session*, earlier versus later.  
- If windows overlap, a random shuffle of windows is invalid.  
- Any statistic used to choose channels, filters, or components must be computed on the training split only.

> **Definition 6.4 (Nested preprocessing).**  
> Estimation of any parameter of $\Phi_{\text{clean}}$ or $\Phi_{\text{feat}}$ inside the training fold only, including spatial filters, component rejection, and channel lists.


In [ ]:
def overlapping_windows(T=1000, win=128, hop=16):
    starts = np.arange(0, T - win + 1, hop)
    return starts


starts = overlapping_windows()
# adjacent windows share 128-16 = 112 samples
shared = 128 - 16
print(f"n windows={starts.size}; samples shared by neighbours={shared}")
print("Shuffling these windows into train/test is leakage, not cross-validation.")


---

## 6.6 Spatial filters: PCA and the two-class idea

A spatial filter $w \in \mathbb{R}^C$ produces $s_t = w^\top v_t$ (Chapter 0). Choosing $w$ is the first place students meet usable linear algebra.

**PCA** on the training covariance $\Sigma$ finds directions of greatest variance. Useful for compression and for spotting a dominant artefact axis. Not, by itself, a class decoder: variance need not be variance *of the paradigm*.

**Common spatial patterns (CSP), idea only.** Given two classes with covariances $\Sigma_+$ and $\Sigma_-$, CSP seeks $w$ that maximise power in one class relative to the other. In one standard form $w$ solves a generalised eigenproblem

$$
\Sigma_+ w = \lambda \Sigma_- w.
$$

The leading filters become the motor-imagery spatial features; log-variance of the projected signals is the usual $z$.

Compute $\Sigma_+$, $\Sigma_-$, and the filters on the **training blocks only**. Applying CSP fitted on all trials is a classic leak and will inflate Chapter 7 numbers.

Implanted arrays may need less spatial invention and more time-series modelling. The geometry does not disappear; it changes scale.


In [ ]:
# Tiny CSP-like illustration in 4-D electrode space.
C, n = 4, 200
# class + : energy on channels 0-1 ; class - : energy on 2-3
A = np.random.randn(n, C) * np.array([1.6, 1.6, 0.4, 0.4])
B = np.random.randn(n, C) * np.array([0.4, 0.4, 1.6, 1.6])
Sig_p = np.cov(A, rowvar=False)
Sig_m = np.cov(B, rowvar=False)
# generalised eigenvectors of Sig_p w = lam Sig_m w
lam, W = np.linalg.eig(np.linalg.pinv(Sig_m) @ Sig_p)
order = np.argsort(np.real(lam))[::-1]
W = np.real(W[:, order])
lam = np.real(lam[order])
print("generalised eigenvalues (class+/class- power):", np.round(lam, 3))
print("first CSP-like filter w:", np.round(W[:, 0], 3))
print("high weights should fall on channels 0-1 for this synthetic pair")


---

## 6.7 Features matched to Chapter 5

**Motor imagery.** Band-pass 8–30 Hz (tune on training data), spatial filter, log-power in 0.5–2 s causal windows. Optional: separate mu and beta, left versus right sites.

**P300.** Time-locked samples after a spatial projection or a small set of midline channels; optionally down-sampled. Do not band-pass so aggressively that the 300 ms bump dies. Features are *shape*, not only power.

**SSVEP / cVEP.** Narrow-band power or correlation with reference flicker / codes. Include harmonics if the display and $f_s$ support them.

**Attempted speech.** Short overlapping windows of high-dimensional arrays; often learned representations (Chapter 7). Hand-crafted bands still appear in papers; they are a baseline you must beat, not a relic you may skip.

**Passive workload.** Often a soup of band ratios and long windows. Demand the same leakage rule and the same nuisance audit. A “theta/beta ratio” is a feature, not a theory of mind.


---

## 6.8 What you must write down

A methods paragraph that cannot be reimplemented is not a methods paragraph. Minimum contents:

- $f_s$, resolution, montage, and reference;  
- filter cut-offs, order, and whether causal;  
- epoch or window length, hop, and timestamp convention;  
- artefact rule, including whether labels entered the rule;  
- spatial-filter type and the split on which it was fit;  
- feature formula in one equation;  
- any normalisation (z-score, whitening) and its training-only statistics.

If two laboratories cannot match your $Z$ from your $X$ and this list, Chapter 7 is theatre.


---

## 6.9 What to carry into Chapter 7

1. Features are designed objects matched to a paradigm.  
2. Causal filters and causal windows belong to interfaces; zero-phase belongs to pretty offline plots.  
3. Spatial filters and channel picks are part of $\theta$ and follow nested preprocessing.  
4. An artefact that classifies is a competing system.  
5. Chapter 7 may replace $\Phi_{\text{feat}}$ with a network. It may not replace honesty about splits.


---

## 6.10 Chapter summary

Traces become features through irreversible maps: reference, filter, artefact rule, window, spatial projection, statistic. Each map destroys something. Name what is destroyed. Motor imagery wants spatially filtered band power; P300 wants time-locked shape; SSVEP wants narrow-band locking; speech wants short multi-channel sequences. All of them want a split in which the future did not tune the past. The live loop must run the same $\Phi$ that the paper named, including its delay.


---

## Exercises

**6.1** Using the referencing cell, explain in four sentences why a blink-contaminated reference can create a false P300 at every site.

**6.2** A pipeline uses `filtfilt` with a 0.1–30 Hz band, then claims online P300 spelling. Identify the contradiction and propose a causal substitute, including where delay will appear in the Chapter 4 budget.

**6.3** Windows of 1 s with 90 percent overlap are shuffled into 5-fold CV for motor imagery. Write the referee paragraph that rejects the accuracy number.

**6.4** Fit PCA on the raw three-channel example (training half only) and interpret the first component. Is it neural?

**6.5** Derive, in words, why CSP fitted on all trials leaks. Then state the nested procedure.

**6.6** Design a nuisance-decoder audit for a forehead dry-EEG imagery study. Which channels, which labels, what result would halt the neural claim?

**6.7** Write the feature equation $z = \ldots$ for (a) two-class imagery after one CSP filter and (b) a single-channel SSVEP power feature at 12 Hz and 24 Hz.

**6.8** Amina’s home EEG has 16 channels, no EOG, and a ceiling fan at 50 Hz. List the $\Phi_{\text{clean}}$ choices you will freeze before her first labelled block.


---

## Annotated reading

- MNE-Python preprocessing tutorial — for the operational counterpart of Sections 6.2–6.4.  
- Blankertz et al. on common spatial patterns — read after the eigenproblem in Section 6.6.  
- A sceptical paper on EEG artefact and “BCI” control — keep it beside every motor-imagery claim.  
- Chapter 0 leakage cell — reread before you touch Chapter 7.

---

*End of Chapter 6.*  
*Next: Chapter 7 — Learning systems for neural data.*
